#03 -- Embeddings and Vector Search Index

In [0]:
CATALOG = 'stock_research_capstone'
SCHEMA = 'main'
EMBEDDING_MODEL_ENDPOINT = 'databricks-bge-large-en'
VECTOR_SEARCH_ENDPOINT = 'vector_search'

SOURCE_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.silver_news"
SOURCE_COMPANIES_TABLE = f"{CATALOG}.{SCHEMA}.silver_companies"
EMBEDDINGS_TABLE = f"{CATALOG}.{SCHEMA}.text_embeddings"
VECTOR_INDEX_NAME = f"{CATALOG}.{SCHEMA}.text_embeddings_index"


print(f"Embedding model endpoint: {EMBEDDING_MODEL_ENDPOINT}")
print(f"Vector search endpoint: {VECTOR_SEARCH_ENDPOINT}")
print(f"Index: {VECTOR_INDEX_NAME}")

In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import mlflow.deployments
from databricks.vector_search.client import VectorSearchClient
import time

# News articles: use title + summary + full_text
df_news_text = (
    spark.table(SOURCE_NEWS_TABLE)
    .select(
        F.concat_ws(" | ", F.col("ticker"), F.col("title")).alias("doc_id"),
        F.lit("news").alias("doc_type"),
        F.col("ticker"),
        F.col("title"),
        F.coalesce(
            F.concat_ws("\n\n", F.col("title"), F.col("summary"), F.col("full_text")),
            F.concat_ws("\n\n", F.col("title"), F.col("summary")),
            F.col("title")
        ).alias("text"),
        F.col("published_at").cast("string").alias("metadata_date"),
        F.col("source").alias("metadata_source")
    )
)

# Company descriptions
df_company_text = (
    spark.table(SOURCE_COMPANIES_TABLE)
    .select(
        F.concat_ws(" | ", F.col("ticker"), F.lit("profile")).alias("doc_id"),
        F.lit("company_profile").alias("doc_type"),
        F.col("ticker"),
        F.col("company_name").alias("title"),
        F.concat_ws(
            "\n",
            F.concat(F.lit("Company: "), F.col("company_name")),
            F.concat(F.lit("Sector: "), F.col("sector")),
            F.concat(F.lit("Industry: "), F.col("industry")),
            F.col("company_description")
        ).alias("text"),
        F.col("updated_at").cast("string").alias("metadata_date"),
        F.lit("company_filing").alias("metadata_source")
    )
)

# Union into single corpus
df_corpus = df_news_text.unionByName(df_company_text)
print(f"Total documents to embed: {df_corpus.count()}")
df_corpus.show(5, truncate=60)


In [0]:
client = mlflow.deployments.get_deploy_client("databricks")

def get_embeddings_batch(texts: list) -> list:
    """Call the embedding endpoint for a batch of texts."""
    response = client.predict(
        endpoint=EMBEDDING_MODEL_ENDPOINT,
        inputs={"input": texts}
    )
    return [item["embedding"] for item in response["data"]]

# Collect texts and generate embeddings in batches
BATCH_SIZE = 20
rows = df_corpus.collect()

all_records = []
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    texts = [row["text"][:8000] for row in batch]  # Truncate to model max

    embeddings = get_embeddings_batch(texts)

    for row, emb in zip(batch, embeddings):
        all_records.append({
            "doc_id": row["doc_id"],
            "doc_type": row["doc_type"],
            "ticker": row["ticker"],
            "title": row["title"],
            "text": row["text"],
            "metadata_date": row["metadata_date"],
            "metadata_source": row["metadata_source"],
            "embedding": emb
        })

    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Processed {min(i + BATCH_SIZE, len(rows))}/{len(rows)} documents")

print(f"\nEmbedded {len(all_records)} documents.")



In [0]:

# Define schema with array of floats for embeddings
embedding_schema = StructType([
    StructField("doc_id", StringType()),
    StructField("doc_type", StringType()),
    StructField("ticker", StringType()),
    StructField("title", StringType()),
    StructField("text", StringType()),
    StructField("metadata_date", StringType()),
    StructField("metadata_source", StringType()),
    StructField("embedding", ArrayType(FloatType()))
])

df_embeddings = spark.createDataFrame(all_records, schema=embedding_schema)
df_embeddings.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EMBEDDINGS_TABLE)

print(f"Embeddings table written: {EMBEDDINGS_TABLE}")
print(f"Embedding dimension: {len(all_records[0]['embedding'])}")
df_embeddings.select("doc_id", "doc_type", "ticker", "title").show(5, truncate=50)


In [0]:
vsc = VectorSearchClient()

# Enable Change Data Feed on the source table (required for Delta Sync)
spark.sql(f"ALTER TABLE {EMBEDDINGS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"✓ Change Data Feed enabled on {EMBEDDINGS_TABLE}")

# Create the index (Delta Sync mode - auto-syncs when table updates)
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_INDEX_NAME,
        source_table_name=EMBEDDINGS_TABLE,
        pipeline_type="TRIGGERED",  # Use TRIGGERED or CONTINUOUS
        primary_key="doc_id",
        embedding_dimension=1024,   # BGE-large dimension; adjust if different model
        embedding_vector_column="embedding"
    )
    print(f"✓ Vector Search index created: {VECTOR_INDEX_NAME}")
    print(f"  Sync mode: TRIGGERED")
    print(f"  Source: {EMBEDDINGS_TABLE}")
    print("  Triggering initial sync...")
    index.sync()
    print("  ✓ Sync triggered")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index {VECTOR_INDEX_NAME} already exists. Syncing...")
        index = vsc.get_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=VECTOR_INDEX_NAME
        )
        index.sync()
    else:
        raise e

In [0]:

# Wait for index to be ready (can take a few minutes after creation)
print("Waiting for index to be ready...")
max_wait = 300  # 5 minutes
wait_interval = 15
elapsed = 0

while elapsed < max_wait:
    try:
        index = vsc.get_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=VECTOR_INDEX_NAME
        )
        status = index.describe().get("status", {}).get("ready", False)
        if status:
            print(f"✓ Index is ready after {elapsed}s")
            break
    except Exception as e:
        if "is not ready" in str(e):
            print(f"  Index status: not ready yet... ({elapsed}s elapsed)")
        else:
            raise e
    time.sleep(wait_interval)
    elapsed += wait_interval
else:
    print(f"⚠ Index not ready after {max_wait}s. It may still be initializing in the background.")
    print("  Try running this cell again in a few minutes.")
    raise RuntimeError("Index not ready - please retry later")

# Semantic search example
test_query = "companies exposed to rising interest rates in the banking sector"
query_embedding = get_embeddings_batch([test_query])[0]

results = index.similarity_search(
    query_vector=query_embedding,
    columns=["doc_id", "doc_type", "ticker", "title", "text"],
    num_results=5
)

print(f"Query: '{test_query}'\n")
print("Top results:")

data = results.get("result", {}).get("data_array", [])
if data:
    for i, row in enumerate(data):
        print(f"  {i+1}. [{row[2]}] {row[3]}")
        print(f"     Type: {row[1]} | Score: {row[-1]:.4f}")
        print()
else:
    print("  No matches found")